# Shared vs Isolated State [Step 07.03 - The subgraph boundary, properly]

> **MLCourse - Agentic AI - LangGraph**

This is the most important notebook in the module. Almost every subgraph bug -
"my subgraph returns nothing", "my key vanished", "KeyError inside a node that
works fine standalone" - is one of the two boundary mistakes taught here.

There are exactly **two ways** to attach a child graph to a parent:

| Mode | How | Requirement | Use when |
|------|-----|-------------|----------|
| **Shared schema** | `parent.add_node("child", compiled_child)` | Parent and child share the key *names* that cross the boundary | The child is part of the same domain and you control both schemas |
| **Isolated schema** | `parent.add_node("child", wrapper_fn)` where the wrapper calls `compiled_child.invoke(...)` | None - you translate explicitly | The child is independent, third-party, reused across domains, or its names clash |

### What you'll learn

- Exactly what LangGraph does to a dict at the boundary (the mental model).
- Three failure demos you can recognise later: **KeyError**, **silent key loss**,
  **reducer surprise**.
- The wrapper-function pattern for isolated state, written properly.
- How to decide which mode to use.

### Key takeaways

- The boundary is a **filter, not a translator**. It passes matching keys and
  drops everything else. It never renames anything for you.
- Shared mode is convenient and coupling-heavy. Isolated mode is explicit and safe.
- When in doubt, use a wrapper. The five extra lines buy you a real interface.

### Setup: environment, model factory, rate-limit-aware call helper


In [ ]:
import os                                  # environment variable access
import time                                # timing + backoff sleeps
from pathlib import Path                   # locating the track root
from dotenv import load_dotenv             # reads KEY=value pairs from .env

# Walk UP from the notebook folder until we find the track root `03_agentic_ai`,
# then load the (gitignored) .env that lives there. Every provider-touching
# notebook in this track uses exactly this block.
TRACK = Path.cwd()
while TRACK.name != "03_agentic_ai" and TRACK != TRACK.parent:
    TRACK = TRACK.parent
load_dotenv(TRACK / ".env")

GROQ_KEY = os.getenv("GROQ_API_KEY")       # never print this value
GROQ_MODEL = "qwen/qwen3.8-27b"            # fast hosted model, generous free tier
OLLAMA_MODEL = "llama3.1:8b"               # local fallback if Groq is unavailable


def make_llm(temperature: float = 0.0, max_tokens: int = 512):
    """Return a chat model. Groq first (fast, hosted); local Ollama as fallback.

    OpenAI is never used anywhere in this course.
    """
    if GROQ_KEY:
        from langchain_groq import ChatGroq
        return ChatGroq(model=GROQ_MODEL, api_key=GROQ_KEY,
                        temperature=temperature, max_tokens=max_tokens)
    from langchain_ollama import ChatOllama
    return ChatOllama(model=OLLAMA_MODEL, temperature=temperature)


def safe_invoke(model, messages, retries: int = 4, pause: float = 1.5):
    """Invoke a chat model with exponential backoff on rate limits (HTTP 429).

    Groq's free tier allows roughly 8000 tokens per minute. Teaching notebooks
    fire many small calls in a row, so a retry loop is not optional here.
    """
    delay = pause
    for attempt in range(retries):
        try:
            out = model.invoke(messages)
            time.sleep(pause)              # pace the next call politely
            return out
        except Exception as exc:
            if attempt == retries - 1:
                raise
            print("  [backoff] %s -- retrying in %.1fs" % (type(exc).__name__, delay))
            time.sleep(delay)
            delay *= 2                     # exponential backoff
    raise RuntimeError("unreachable")


print("Track root :", TRACK.name)
print("Provider   :", "Groq / " + GROQ_MODEL if GROQ_KEY else "Ollama / " + OLLAMA_MODEL)


### 1. The mental model

When a compiled child is used directly as a node, two filters run:

```
parent state ---> [ keep only keys present in the CHILD schema ] ---> child runs
child output ---> [ keep only keys present in the PARENT schema ] ---> merged into parent
```

Both filters are **name-based**. There is no type checking and no renaming.
Say that out loud once: *the boundary matches on key names only.*

Let's prove it.

In [2]:
from typing import Annotated, TypedDict
import operator
from langgraph.graph import StateGraph, START, END


class ChildSchema(TypedDict):
    payload: str            # child reads this
    result: str             # child writes this
    scratch: str            # child-private


def child_node(state: ChildSchema) -> dict:
    return {"result": "processed(" + state["payload"] + ")", "scratch": "internal"}


cg = StateGraph(ChildSchema)
cg.add_node("work", child_node)
cg.add_edge(START, "work")
cg.add_edge("work", END)
child_app = cg.compile()

print("Child alone:", child_app.invoke({"payload": "abc"}))

Child alone: {'payload': 'abc', 'result': 'processed(abc)', 'scratch': 'internal'}


### 2. Failure demo A - the KeyError

Parent calls its data `text`. Child expects `payload`. Names differ, so the child
receives **nothing** for `payload` and blows up on the very first read.

This is the friendliest failure, because at least it is loud.

In [3]:
class BadParent(TypedDict):
    text: str               # NOT called 'payload'
    result: str


bad = StateGraph(BadParent)
bad.add_node("child", child_app)       # direct embed, mismatched input name
bad.add_edge(START, "child")
bad.add_edge("child", END)
bad_app = bad.compile()

try:
    bad_app.invoke({"text": "abc"})
except Exception as exc:
    print("BOOM ->", type(exc).__name__)
    print(str(exc).strip().splitlines()[0])
    print()
    print("Diagnosis: the parent has no key named 'payload', so the child's input")
    print("filter passed an empty dict and `state['payload']` failed.")

BOOM -> KeyError
'payload'

Diagnosis: the parent has no key named 'payload', so the child's input
filter passed an empty dict and `state['payload']` failed.


### 3. Failure demo B - the SILENT one (this is the dangerous one)

Now flip it. The parent *does* supply the input key, but names the **output** key
differently. The child runs perfectly. The parent's output filter drops `result`
because `result` is not in the parent schema. You get no error, no warning - just
a missing value that surfaces three nodes later as `KeyError` or, worse, as an
empty string in your final answer.

In [4]:
class SilentParent(TypedDict):
    payload: str            # input name matches - child runs fine
    outcome: str            # but the child writes 'result', not 'outcome'
    note: str


def read_outcome(state: SilentParent) -> dict:
    got = state.get("outcome", "<MISSING>")
    return {"note": "downstream node saw outcome=" + repr(got)}


silent = StateGraph(SilentParent)
silent.add_node("child", child_app)
silent.add_node("read_outcome", read_outcome)
silent.add_edge(START, "child")
silent.add_edge("child", "read_outcome")
silent.add_edge("read_outcome", END)

out = silent.compile().invoke({"payload": "abc"})
print("No exception was raised. Final state:")
for k, v in sorted(out.items()):
    print("   %-9s = %r" % (k, v))
print()
print("The child DID compute 'processed(abc)'. It was discarded at the boundary")
print("because the parent schema has no key called 'result'.")

No exception was raised. Final state:
   note      = "downstream node saw outcome='<MISSING>'"
   payload   = 'abc'

The child DID compute 'processed(abc)'. It was discarded at the boundary
because the parent schema has no key called 'result'.


> **This is THE pitfall of subgraphs.** Nothing crashes. Nothing logs. A key you
> care about is quietly dropped because two schemas disagreed about a name.
>
> **How to catch it early:**
> 1. After wiring a subgraph, `print(sorted(result.keys()))` once and eyeball it.
> 2. Write down the boundary contract as a comment above `add_node`.
> 3. Or - better - stop relying on name luck and use a wrapper (section 5).

### 4. Failure demo C - the reducer surprise

Shared keys share **reducers too**. If the parent declares
`Annotated[list, operator.add]` and the child returns a list, the child's list is
*appended*, not assigned. People expect replacement and get accumulation, or
they run the graph twice and wonder why everything is duplicated.

In [5]:
class ReducerChild(TypedDict):
    items: Annotated[list, operator.add]


def emit(state: ReducerChild) -> dict:
    return {"items": ["x", "y"]}


rc = StateGraph(ReducerChild)
rc.add_node("emit", emit)
rc.add_edge(START, "emit")
rc.add_edge("emit", END)
reducer_child = rc.compile()


class ReducerParent(TypedDict):
    items: Annotated[list, operator.add]     # SAME name, SAME reducer


def seed(state: ReducerParent) -> dict:
    return {"items": ["seed"]}


rp = StateGraph(ReducerParent)
rp.add_node("seed", seed)
rp.add_node("child", reducer_child)
rp.add_edge(START, "seed")
rp.add_edge("seed", "child")
rp.add_edge("child", END)

print("result:", rp.compile().invoke({"items": []})["items"])
print()
print("The child ran ONCE and emitted ['x','y'], but the parent's reducer appended")
print("them to the seed. If you expected the child to REPLACE items, you now have a")
print("bug that only shows up on the second run.")

result: ['seed', 'seed', 'x', 'y']

The child ran ONCE and emitted ['x','y'], but the parent's reducer appended
them to the seed. If you expected the child to REPLACE items, you now have a
bug that only shows up on the second run.


### 5. The fix: isolated state with a wrapper function

Instead of handing the compiled child to `add_node`, hand it a small function.
That function is the **interface**: it decides what goes in, calls the child,
and decides what comes out.

Five lines. Total control. No name coupling.

In [6]:
class CleanParent(TypedDict):
    text: str               # parent's own vocabulary
    outcome: str            # parent's own vocabulary
    note: str


def child_wrapper(state: CleanParent) -> dict:
    """Adapter node: parent vocabulary -> child vocabulary -> parent vocabulary.

    This function IS the boundary contract, written down in code.
    """
    child_input = {"payload": state["text"]}          # 1. map IN  (explicit rename)
    child_out = child_app.invoke(child_input)         # 2. run the child
    return {"outcome": child_out["result"]}           # 3. map OUT (explicit rename)


def read_outcome2(state: CleanParent) -> dict:
    return {"note": "downstream node saw outcome=" + repr(state["outcome"])}


clean = StateGraph(CleanParent)
clean.add_node("child", child_wrapper)                # a FUNCTION, not the graph
clean.add_node("read_outcome", read_outcome2)
clean.add_edge(START, "child")
clean.add_edge("child", "read_outcome")
clean.add_edge("read_outcome", END)

out = clean.compile().invoke({"text": "abc"})
for k, v in sorted(out.items()):
    print("   %-9s = %r" % (k, v))

   note      = "downstream node saw outcome='processed(abc)'"
   outcome   = 'processed(abc)'
   text      = 'abc'


Everything arrived. And note the extra benefits you get for free with a wrapper:

- **Validation** - reject bad input before the child sees it.
- **Defaults** - supply child keys the parent doesn't have.
- **Error handling** - catch child exceptions and turn them into parent state.
- **Partial output** - take only the fields you want.
- **Independence** - the child's schema can change without touching the parent,
  as long as the wrapper is updated.

Here is the production-shaped version of the same wrapper.

In [7]:
class RobustParent(TypedDict):
    text: str
    outcome: str
    error: str


def robust_wrapper(state: RobustParent) -> dict:
    """A boundary you would actually ship."""
    text = (state.get("text") or "").strip()
    if not text:                                       # validate
        return {"outcome": "", "error": "empty input, child not called"}

    try:
        child_out = child_app.invoke({"payload": text})
    except Exception as exc:                           # contain child failures
        return {"outcome": "", "error": "child failed: %s" % type(exc).__name__}

    return {"outcome": child_out.get("result", ""), "error": ""}   # project output


rb = StateGraph(RobustParent)
rb.add_node("child", robust_wrapper)
rb.add_edge(START, "child")
rb.add_edge("child", END)
rb_app = rb.compile()

print("good input :", rb_app.invoke({"text": "hello"}))
print("empty input:", rb_app.invoke({"text": "   "}))

good input : {'text': 'hello', 'outcome': 'processed(hello)', 'error': ''}
empty input: {'text': '   ', 'outcome': '', 'error': 'empty input, child not called'}


### 6. A real, LLM-backed isolated subgraph

Let's do it once with a model in the loop, so the pattern is not just toy dicts.

The child is a **translator** with its own vocabulary (`source_text`, `target_lang`,
`translation`). The parent is an **article pipeline** with vocabulary
(`headline`, `language`, `localised`). Nothing matches, and that is fine.

In [8]:
class TranslatorState(TypedDict):
    source_text: str
    target_lang: str
    translation: str


def translate(state: TranslatorState) -> dict:
    prompt = ("Translate the following text into %s. "
              "Output ONLY the translation, no commentary.\n\n%s"
              % (state["target_lang"], state["source_text"]))
    return {"translation": safe_invoke(t_llm, prompt).content.strip()}


t_llm = make_llm(max_tokens=80)

tg = StateGraph(TranslatorState)
tg.add_node("translate", translate)
tg.add_edge(START, "translate")
tg.add_edge("translate", END)
translator_app = tg.compile()

# The child is fully usable on its own, in its OWN vocabulary:
print(translator_app.invoke({"source_text": "Good morning, team.",
                             "target_lang": "French"})["translation"])

Bonjour, l'équipe.


In [9]:
class ArticleState(TypedDict):
    headline: str
    language: str
    localised: str
    status: str


def translate_boundary(state: ArticleState) -> dict:
    """Adapter: article vocabulary <-> translator vocabulary."""
    child_out = translator_app.invoke({
        "source_text": state["headline"],       # headline -> source_text
        "target_lang": state["language"],       # language -> target_lang
    })
    return {"localised": child_out["translation"]}   # translation -> localised


def publish(state: ArticleState) -> dict:
    return {"status": "published '%s' in %s" % (state["localised"], state["language"])}


ag = StateGraph(ArticleState)
ag.add_node("translate", translate_boundary)
ag.add_node("publish", publish)
ag.add_edge(START, "translate")
ag.add_edge("translate", "publish")
ag.add_edge("publish", END)

res = ag.compile().invoke({"headline": "LangGraph ships stateful agents",
                           "language": "Spanish"})
print("localised:", res["localised"])
print("status   :", res["status"])

localised: LangGraph incluye agentes con estado.
status   : published 'LangGraph incluye agentes con estado.' in Spanish


### 7. Decision guide

Ask these in order:

1. **Do the parent and child belong to the same domain and share an owner?**
   No -> **isolated (wrapper)**. Stop here.
2. **Do the crossing keys already have identical names and identical reducers?**
   No -> **isolated (wrapper)**.
3. **Is the child reused by parents with different vocabularies?**
   Yes -> **isolated (wrapper)**.
4. **Is the crossing state literally `messages` with the same `add_messages` reducer?**
   Yes -> **shared** is idiomatic and fine (this is how agent-as-subgraph works).

In practice: **`messages`-carrying agent subgraphs -> shared. Everything else -> wrapper.**

### 8. Debug checklist

When a subgraph "does nothing":

- [ ] `print(sorted(child_app.invoke(minimal_input).keys()))` - does the child work alone?
- [ ] Compare `set(ParentState.__annotations__) & set(ChildState.__annotations__)` -
      is the crossing set what you think it is?
- [ ] `parent_app.stream(..., subgraphs=True, stream_mode="updates")` - did the child
      run at all, and what did it emit?
- [ ] Is a shared key annotated with a reducer that is appending when you wanted replacing?

In [10]:
# The one-liner that would have caught failure demos A and B instantly:
shared = set(SilentParent.__annotations__) & set(ChildSchema.__annotations__)
print("ChildSchema  :", sorted(ChildSchema.__annotations__))
print("SilentParent :", sorted(SilentParent.__annotations__))
print("crossing keys:", sorted(shared) or "<none>")
print()
print("'result' in crossing set?", "result" in shared,
      "  <- that is why the output disappeared")

ChildSchema  : ['payload', 'result', 'scratch']
SilentParent : ['note', 'outcome', 'payload']
crossing keys: ['payload']

'result' in crossing set? False   <- that is why the output disappeared


### Recap

- The subgraph boundary **filters by key name** in both directions. It never renames.
- Mismatched *input* names -> loud `KeyError`. Mismatched *output* names -> **silent loss**.
- Shared keys inherit the parent's **reducer**, which can turn assignment into accumulation.
- The wrapper-function pattern gives you an explicit, testable, validated interface.

### Next

**[04_send_api_map_reduce](04_send_api_map_reduce.ipynb)** - fan out to N branches
when N is only known at runtime.